# Week 7 Day 2: Guided Practice - CIFAR-10 Regularization

**Dataset:** CIFAR-10 (3 classes: airplane, car, bird)
**Time:** 20 minutes (instructor-led)
**Scaffolding:** 70% provided

---

## Goal

Apply Day 1's regularization workflow to **color images** - a harder problem than Fashion-MNIST.

**Workflow:** Load data -> Build baseline -> Diagnose overfitting -> Add regularization -> Compare

---

## Part 1: Setup and Data Loading (Provided)

In [ ]:
# Setup (provided)
import os
os.environ['KERAS_BACKEND'] = 'torch'

import keras
from keras import layers
from keras.callbacks import EarlyStopping
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

print(f"Keras version: {keras.__version__}")
print("Setup complete!")

In [ ]:
# Load CIFAR-10 and select 3 classes (provided)
from keras.datasets import cifar10

(X_train_full, y_train_full), (X_test_full, y_test_full) = cifar10.load_data()
y_train_full = y_train_full.flatten()
y_test_full = y_test_full.flatten()

# Select 3 classes: 0=airplane, 1=car, 2=bird
selected_classes = [0, 1, 2]
class_names = ['Airplane', 'Car', 'Bird']

# Filter training data
train_mask = np.isin(y_train_full, selected_classes)
X_train_raw = X_train_full[train_mask]
y_train_raw = y_train_full[train_mask]

# Filter test data
test_mask = np.isin(y_test_full, selected_classes)
X_test = X_test_full[test_mask]
y_test = y_test_full[test_mask]

print(f"Training samples: {X_train_raw.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Image shape: {X_train_raw.shape[1:]} (32x32 color!)")
print(f"Classes: {class_names}")

In [ ]:
# Visualize some samples (provided)
plt.figure(figsize=(12, 3))
for i, cls in enumerate(selected_classes):
    idx = np.where(y_train_raw == cls)[0][:3]
    for j, sample_idx in enumerate(idx):
        plt.subplot(1, 9, i * 3 + j + 1)
        plt.imshow(X_train_raw[sample_idx])
        plt.title(class_names[i], fontsize=9)
        plt.axis('off')
plt.suptitle('CIFAR-10 Samples (32x32 COLOR images)', fontsize=12)
plt.tight_layout()
plt.show()

print("\nNotice: These are MUCH harder than Fashion-MNIST!")
print("Color images with complex backgrounds and varying poses.")

In [ ]:
# Normalize and prepare data (provided)

# Normalize to 0-1
X_train_raw = X_train_raw / 255.0
X_test = X_test / 255.0

# Three-way split
X_train, X_val, y_train, y_val = train_test_split(
    X_train_raw, y_train_raw,
    test_size=0.2, random_state=42, stratify=y_train_raw
)

# Flatten for Dense layers: 32x32x3 = 3072 features
X_train_flat = X_train.reshape(-1, 3072)
X_val_flat = X_val.reshape(-1, 3072)
X_test_flat = X_test.reshape(-1, 3072)

print(f"Training: {X_train_flat.shape}")
print(f"Validation: {X_val_flat.shape}")
print(f"Test: {X_test_flat.shape}")
print(f"\nFeatures per image: {X_train_flat.shape[1]} (vs 784 for Fashion-MNIST!)")

---

## Part 2: Build Baseline Model (No Regularization)

**YOUR TASK:** Build a model WITHOUT Dropout to see how badly it overfits.

**Architecture:**
- Input: 3072 features
- Dense: 512 neurons, relu
- Dense: 256 neurons, relu
- Dense: 128 neurons, relu
- Output: 3 neurons, softmax

In [ ]:
# TODO: Build baseline model (no Dropout!)

keras.utils.set_random_seed(42)  # provided: keeps everyone's results comparable

model_baseline = keras.Sequential([
    layers.Input(shape=(____,)),
    layers.Dense(____, activation='____'),
    layers.Dense(____, activation='____'),
    layers.Dense(____, activation='____'),
    layers.Dense(____, activation='____')
])

model_baseline.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_baseline.summary()

In [ ]:
# Train baseline (provided - just run this cell)
print("Training baseline model (no regularization)...")
history_baseline = model_baseline.fit(
    X_train_flat, y_train,
    validation_data=(X_val_flat, y_val),
    epochs=50,
    batch_size=128,
    verbose=1
)
print("\nBaseline training complete!")

In [ ]:
# Plot baseline curves (provided)
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_baseline.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history_baseline.history['val_loss'], label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Baseline: Loss (NO Regularization)')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_baseline.history['accuracy'], label='Training Accuracy', linewidth=2)
plt.plot(history_baseline.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Baseline: Accuracy (NO Regularization)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

train_acc = history_baseline.history['accuracy'][-1]
val_acc = history_baseline.history['val_accuracy'][-1]
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Gap: {(train_acc - val_acc):.4f}")

### PAUSE HERE - Discuss with Instructor

**Questions to answer:**
1. What pattern do you see in the curves?
2. How does the gap compare to what you saw with Fashion-MNIST on Day 1?
3. Why might CIFAR-10 overfit MORE than Fashion-MNIST?

---

## Part 3: Build Regularized Model

**YOUR TASK:** Add Dropout layers and configure EarlyStopping to fix the overfitting.

**Architecture (same as baseline + Dropout):**
- Input: 3072 features
- Dense: 512 neurons, relu
- **Dropout: 0.2**
- Dense: 256 neurons, relu
- **Dropout: 0.2**
- Dense: 128 neurons, relu
- **Dropout: 0.2**
- Output: 3 neurons, softmax

> **Tip:** Start with 0.2. Dropout rate is a hyperparameter you tune - if accuracy drops on BOTH training and validation, you've over-regularized (underfitting).

In [ ]:
# TODO: Build regularized model (add Dropout after each Dense layer)

keras.utils.set_random_seed(42)  # provided: same seed as baseline -> fair comparison

model_regularized = keras.Sequential([
    layers.Input(shape=(3072,)),
    layers.Dense(512, activation='relu'),
    layers.Dropout(____),              # TODO: What rate?
    layers.Dense(256, activation='relu'),
    layers.Dropout(____),              # TODO: What rate?
    layers.Dense(128, activation='relu'),
    layers.Dropout(____),              # TODO: What rate?
    layers.Dense(3, activation='softmax')
])

model_regularized.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Regularized model built!")
model_regularized.summary()

In [ ]:
# TODO: Configure EarlyStopping

early_stop = EarlyStopping(
    monitor='____',              # TODO: What metric to watch?
    patience=____,               # TODO: How many epochs to wait?
    restore_best_weights=____    # TODO: True or False?
)

print("EarlyStopping configured!")

In [ ]:
# TODO: Train regularized model with EarlyStopping

print("Training regularized model...")
history_regularized = model_regularized.fit(
    X_train_flat, y_train,
    validation_data=(X_val_flat, y_val),
    epochs=60,
    batch_size=128,
    callbacks=[____],           # TODO: Pass the callback
    verbose=1
)
print("\nRegularized training complete!")

---

## Part 4: Compare Results

**YOUR TASK:** Plot both models side-by-side and analyze the improvement.

In [ ]:
# Side-by-side comparison (provided)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Baseline curves
axes[0].plot(history_baseline.history['accuracy'], label='Train Acc', linewidth=2)
axes[0].plot(history_baseline.history['val_accuracy'], label='Val Acc', linewidth=2)
axes[0].set_title('Baseline (No Regularization)', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Regularized curves
axes[1].plot(history_regularized.history['accuracy'], label='Train Acc', linewidth=2)
axes[1].plot(history_regularized.history['val_accuracy'], label='Val Acc', linewidth=2)
axes[1].set_title('Regularized (Dropout + EarlyStopping)', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Quantitative comparison (provided)
baseline_train = history_baseline.history['accuracy'][-1]
baseline_val = history_baseline.history['val_accuracy'][-1]
baseline_gap = baseline_train - baseline_val

# EarlyStopping restored the BEST weights (lowest val_loss), which is usually NOT
# the final epoch. Report metrics from that restored epoch for a fair comparison.
best_epoch = int(np.argmin(history_regularized.history['val_loss']))
reg_train = history_regularized.history['accuracy'][best_epoch]
reg_val = history_regularized.history['val_accuracy'][best_epoch]
reg_gap = reg_train - reg_val

print("=" * 50)
print("MODEL COMPARISON")
print("=" * 50)
print(f"{'Metric':<25} {'Baseline':>10} {'Regularized':>12}")
print("-" * 50)
print(f"{'Training Accuracy':<25} {baseline_train:>10.4f} {reg_train:>12.4f}")
print(f"{'Validation Accuracy':<25} {baseline_val:>10.4f} {reg_val:>12.4f}")
print(f"{'Gap (overfit indicator)':<25} {baseline_gap:>10.4f} {reg_gap:>12.4f}")
print("-" * 50)
print(f"{'Epochs Trained':<25} {len(history_baseline.history['loss']):>10} {len(history_regularized.history['loss']):>12}")
print(f"{'Gap Reduction':<25} {'':<10} {((baseline_gap - reg_gap) / baseline_gap * 100):>11.1f}%")
print("=" * 50)

In [ ]:
# Final test evaluation (provided)
test_loss, test_acc = model_regularized.evaluate(X_test_flat, y_test, verbose=0)
print(f"\nFinal Test Accuracy: {test_acc:.4f}")
print(f"Validation Accuracy: {reg_val:.4f}")
print(f"Test-Val Difference: {abs(test_acc - reg_val):.4f}")

---

## Discussion Questions

**Answer with your neighbor or as a class:**

1. **How did CIFAR-10 overfitting compare to Fashion-MNIST?**
   - Was the gap bigger or smaller? Why?

2. **Did the same regularization techniques work?**
   - You used Dropout (0.2 here) + EarlyStopping - the same *tools* as Day 1
   - Did the same *rate* transfer? What happens if you try 0.3 instead? (Hint: watch for underfitting)
   - What does this tell you about transferable tools vs. settings you tune per dataset?

3. **Why is CIFAR-10 harder than Fashion-MNIST?**
   - Think about: image complexity, color channels, background variation
   - What would you need to do better on CIFAR-10? (Hint: Week 8 might help!)

---

*Week 7 Day 2 Guided Practice | Version 1.0 | March 2026*